# 01 — Data Exploration
Explore the DTD dataset structure, class distribution, and generated heightmaps.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

In [ ]:
# ── Download DTD if not present ──
DTD_ROOT = '../data/dtd'
if not Path(DTD_ROOT).exists():
    print('Downloading DTD ...')
    os.system('python ../data/download_dtd.py --dest ../data')
else:
    print('DTD found.')

In [ ]:
# ── Dataset statistics ──
images_dir = Path(DTD_ROOT) / 'images'
categories = sorted([d.name for d in images_dir.iterdir() if d.is_dir()])
counts = {c: len(list((images_dir / c).glob('*.jpg'))) for c in categories}

print(f'Categories: {len(categories)}')
print(f'Total images: {sum(counts.values())}')
print(f'Images per class: {list(counts.values())[0]} (uniform)')

In [ ]:
# ── Class distribution bar chart ──
fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(list(counts.keys()), list(counts.values()), color='steelblue')
ax.set_xticklabels(list(counts.keys()), rotation=90, fontsize=8)
ax.set_ylabel('Image count')
ax.set_title('DTD — Images per Category')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualize sample textures ──
sample_cats = random.sample(categories, 12)
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for ax, cat in zip(axes.flat, sample_cats):
    imgs = list((images_dir / cat).glob('*.jpg'))
    img = Image.open(random.choice(imgs)).resize((200, 200))
    ax.imshow(img)
    ax.set_title(cat, fontsize=9)
    ax.axis('off')
plt.suptitle('Random DTD Texture Samples', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Generate and compare heightmaps ──
from data.heightmap_generator import heightmap_from_path

cat = random.choice(categories)
img_path = random.choice(list((images_dir / cat).glob('*.jpg')))
tex_img = Image.open(img_path)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(tex_img); axes[0].set_title('Original Texture'); axes[0].axis('off')

for i, sigma in enumerate([1.0, 2.0, 4.0]):
    hm = heightmap_from_path(str(img_path), sigma=sigma)
    axes[i+1].imshow(hm, cmap='gray')
    axes[i+1].set_title(f'Heightmap σ={sigma}')
    axes[i+1].axis('off')

plt.suptitle(f'Texture → Heightmap  ({cat})', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Height distribution analysis ──
import numpy as np
from data.heightmap_generator import heightmap_from_path

hm = heightmap_from_path(str(img_path), sigma=2.0)
arr = np.array(hm)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(arr, cmap='hot')
axes[0].set_title('Heightmap (hot colormap)')
axes[0].axis('off')

axes[1].hist(arr.flatten(), bins=64, color='steelblue', edgecolor='none')
axes[1].set_xlabel('Height value (0–255)')
axes[1].set_ylabel('Pixel count')
axes[1].set_title(f'Height Distribution  (mean={arr.mean():.1f}, std={arr.std():.1f})')
plt.tight_layout()
plt.show()